In [ ]:
import zipfile
import os

zip_path = "fiw_embeddings.zip"
extract_path = "data"

# Extract precomputed face embeddings (ArcFace features)
# Each family folder contains embeddings for its members
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print(os.listdir(extract_path))

['FIDs', 'fiw_embeddings']


In [27]:
print(os.listdir("data/fiw_embeddings"))

['F0001', 'F0002', 'F0003', 'F0004', 'F0005', 'F0006', 'F0007', 'F0008', 'F0009', 'F0010', 'F0011', 'F0012', 'F0013', 'F0015', 'F0016', 'F0017', 'F0018', 'F0019', 'F0020', 'F0021', 'F0022', 'F0023', 'F0024', 'F0025', 'F0026', 'F0027', 'F0028', 'F0029', 'F0030', 'F0031', 'F0032', 'F0033', 'F0034', 'F0035', 'F0036', 'F0037', 'F0038', 'F0039', 'F0040', 'F0041', 'F0042', 'F0043', 'F0044', 'F0045', 'F0046', 'F0047', 'F0048', 'F0049', 'F0050', 'F0051', 'F0052', 'F0053', 'F0054', 'F0055', 'F0058', 'F0059', 'F0060', 'F0061', 'F0062', 'F0063', 'F0064', 'F0065', 'F0066', 'F0067', 'F0068', 'F0069', 'F0070', 'F0071', 'F0072', 'F0073', 'F0074', 'F0075', 'F0076', 'F0077', 'F0078', 'F0079', 'F0080', 'F0081', 'F0082', 'F0083', 'F0084', 'F0085', 'F0086', 'F0087', 'F0088', 'F0089', 'F0090', 'F0091', 'F0092', 'F0093', 'F0094', 'F0095', 'F0096', 'F0097', 'F0098', 'F0099', 'F0100', 'F0101', 'F0102', 'F0103', 'F0104', 'F0105', 'F0106', 'F0107', 'F0108', 'F0109', 'F0110', 'F0111', 'F0112', 'F0113', 'F0114', 

In [28]:
for root, dirs, files in os.walk("data/fiw_embeddings"):
    print(root)
    break

data/fiw_embeddings


In [29]:
family_id = "F0001"
family_path = f"data/fiw_embeddings/{family_id}"

print(os.listdir(family_path)[:20])

['MID1', 'MID2', 'MID3', 'MID4']


In [30]:
for root, dirs, files in os.walk(family_path):
    print("ROOT:", root)
    print("DIRS:", dirs[:10])
    print("FILES:", files[:10])
    print()
    
    if root != family_path:
        break

ROOT: data/fiw_embeddings/F0001
DIRS: ['MID1', 'MID2', 'MID3', 'MID4']
FILES: []

ROOT: data/fiw_embeddings/F0001\MID1
DIRS: []
FILES: ['mean_embedding.pkl', 'P00001_face2.pkl', 'P00002_face3.pkl', 'P00003_face1.pkl', 'P00004_face3.pkl', 'P00007_face2.pkl', 'P00008_face7.pkl']



In [ ]:
import pickle
import numpy as np

family_id = "F0001"
family_path = f"data/fiw_embeddings/{family_id}"

# Build node features:
# Each node corresponds to a person (MID)
# Feature = mean ArcFace embedding (512-dim)
node_features = []
node_to_idx = {}

for idx, mid in enumerate(os.listdir(family_path)):
    mid_path = os.path.join(family_path, mid)
    emb_path = os.path.join(mid_path, "mean_embedding.pkl")
    
    if os.path.exists(emb_path):
        with open(emb_path, "rb") as f:
            emb = pickle.load(f)
        
        node_features.append(emb)
        node_to_idx[mid] = idx

# numpy → torch
import torch
x = torch.tensor(np.array(node_features), dtype=torch.float)

print("x shape:", x.shape)
print("node_to_idx:", node_to_idx)

x shape: torch.Size([4, 1, 512])
node_to_idx: {'MID1': 0, 'MID2': 1, 'MID3': 2, 'MID4': 3}


In [32]:
fids_zip = "FIDs.zip"

with zipfile.ZipFile(fids_zip, "r") as zip_ref:
    zip_ref.extractall("data")

print(os.listdir("data"))

['FIDs', 'fiw_embeddings']


In [ ]:
import pandas as pd
import torch
import os

family_id = "F0001"
mid_csv_path = f"data/FIDs/{family_id}/mid.csv"

mid_df = pd.read_csv(mid_csv_path)

edge_list = []

# Build graph edges from FIW relationship matrix
# If RID != 0, a relationship exists between two individuals
for _, row in mid_df.iterrows():
    src_mid = f"MID{int(row['MID'])}"
    
    for col in mid_df.columns:
        if col.isdigit():   # relationship matrix column: 1,2,3,4...
            rid = int(row[col])
            if rid != 0:
                dst_mid = f"MID{int(col)}"
                
                if src_mid in node_to_idx and dst_mid in node_to_idx:
                    edge_list.append([node_to_idx[src_mid], node_to_idx[dst_mid]])

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

print(edge_index)
print(edge_index.shape)

tensor([[0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3],
        [1, 2, 3, 0, 2, 3, 0, 1, 3, 0, 1, 2]])
torch.Size([2, 12])


In [34]:
%pip install torch-geometric

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
x = x.squeeze(1)  # [4, 1, 512] -> [4, 512]

from torch_geometric.data import Data

# Create graph data object for PyTorch Geometric
# x: node features, edge_index: family connections
data = Data(x=x, edge_index=edge_index)

print(data)
print("x:", data.x.shape)
print("edge_index:", data.edge_index.shape)

Data(x=[4, 512], edge_index=[2, 12])
x: torch.Size([4, 512])
edge_index: torch.Size([2, 12])


In [36]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, DeepGraphInfomax

In [ ]:
# GCN encoder:
# Learns node representations by aggregating neighbor information
class Encoder(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv = GCNConv(in_channels, hidden_channels)

    def forward(self, x, edge_index):
        x = self.conv(x, edge_index)
        return F.relu(x)

In [ ]:
# Corrupt node features by random shuffling
# Used to generate negative samples for DGI training
def corruption(x, edge_index):
    perm = torch.randperm(x.size(0))
    return x[perm], edge_index

In [ ]:
# Deep Graph Infomax (DGI):
# Self-supervised model that maximizes mutual information
# between node embeddings and global graph representation
model = DeepGraphInfomax(
    hidden_channels=128,
    encoder=Encoder(in_channels=512, hidden_channels=128),
    summary=lambda z, *args, **kwargs: torch.sigmoid(z.mean(dim=0)),
    corruption=corruption
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

data = data.to(device)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

cuda


In [ ]:
model.train()

# Train DGI model using contrastive objective
for epoch in range(200):
    optimizer.zero_grad()
    
    pos_z, neg_z, summary = model(data.x, data.edge_index)
    loss = model.loss(pos_z, neg_z, summary)
    
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.3864
Epoch 20, Loss: 1.3866
Epoch 40, Loss: 1.3863
Epoch 60, Loss: 1.3863
Epoch 80, Loss: 1.3863
Epoch 100, Loss: 1.3863
Epoch 120, Loss: 1.3863
Epoch 140, Loss: 1.3863
Epoch 160, Loss: 1.3863
Epoch 180, Loss: 1.3863


In [41]:
model.eval()
with torch.no_grad():
    z = model.encoder(data.x, data.edge_index)

print(z.shape)

torch.Size([4, 128])


In [ ]:
# Build a global graph by merging all family graphs
# This enables learning embeddings across all families
all_node_features = []
all_edge_list = []
node_offset = 0

all_family_ids = os.listdir("data/fiw_embeddings")

global_node_to_idx = {}

for family_id in all_family_ids:
    family_path = f"data/fiw_embeddings/{family_id}"
    mid_list = os.listdir(family_path)

    local_node_to_idx = {}

    # node
    for i, mid in enumerate(mid_list):
        emb_path = os.path.join(family_path, mid, "mean_embedding.pkl")
        if os.path.exists(emb_path):
            with open(emb_path, "rb") as f:
                emb = pickle.load(f)

            all_node_features.append(emb)
            local_node_to_idx[mid] = node_offset
            global_node_to_idx[(family_id, mid)] = node_offset
            node_offset += 1

    # edge
    mid_csv_path = f"data/FIDs/{family_id}/mid.csv"
    if not os.path.exists(mid_csv_path):
        continue

    df = pd.read_csv(mid_csv_path)

    for _, row in df.iterrows():
        src_mid = f"MID{int(row['MID'])}"

        for col in df.columns:
            if col.isdigit():
                if int(row[col]) != 0:
                    dst_mid = f"MID{int(col)}"

                    if src_mid in local_node_to_idx and dst_mid in local_node_to_idx:
                        u = local_node_to_idx[src_mid]
                        v = local_node_to_idx[dst_mid]
                        all_edge_list.append([u, v])

# tensor
x = torch.tensor(np.array(all_node_features), dtype=torch.float).squeeze(1)
edge_index = torch.tensor(all_edge_list, dtype=torch.long).t().contiguous()

data = Data(x=x, edge_index=edge_index)

print(data)

Data(x=[5773, 512], edge_index=[2, 19023])


In [43]:
node_family = []

for (family_id, mid), idx in global_node_to_idx.items():
    node_family.append(family_id)

node_family = np.array(node_family)

In [ ]:
import random
import torch.nn.functional as F

# Node-level contrastive loss:
# - Pull nodes from the same family closer
# - Push nodes from different families apart
# Used for indexing / retrieval purpose
def contrastive_loss(z, node_family, num_samples=100):
    loss = 0

    for _ in range(num_samples):
        i = random.randint(0, len(z)-1)
        
        # positive
        same_family = np.where(node_family == node_family[i])[0]
        j = int(np.random.choice(same_family))
        
        # negative
        diff_family = np.where(node_family != node_family[i])[0]
        k = int(np.random.choice(diff_family))

        zi = z[i]
        zj = z[j]
        zk = z[k]

        pos_sim = F.cosine_similarity(zi, zj, dim=0)
        neg_sim = F.cosine_similarity(zi, zk, dim=0)

        loss += -torch.log(torch.sigmoid(pos_sim - neg_sim))

    return loss / num_samples

In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

data = data.to(device)
model = model.to(device)

print(data.x.device)
print(next(model.parameters()).device)

cuda
cuda:0
cuda:0


In [ ]:
model.train()

for epoch in range(200):
    optimizer.zero_grad()

    pos_z, neg_z, summary = model(data.x, data.edge_index)
    dgi_loss = model.loss(pos_z, neg_z, summary)

    z = model.encoder(data.x, data.edge_index)
    cont_loss = contrastive_loss(z, node_family)

    # Combine DGI loss (structure learning) and contrastive loss (family clustering)
    loss = dgi_loss + cont_loss 

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.9218
Epoch 20, Loss: 1.7054
Epoch 40, Loss: 1.3702
Epoch 60, Loss: 1.1751
Epoch 80, Loss: 1.0142
Epoch 100, Loss: 0.9632
Epoch 120, Loss: 0.8920
Epoch 140, Loss: 0.8094
Epoch 160, Loss: 0.8039
Epoch 180, Loss: 0.7919


In [ ]:
model.eval()

with torch.no_grad():
    z = model.encoder(data.x, data.edge_index)

# Save final embeddings for vector search (indexing stage)
torch.save({
    "embeddings": z.cpu(),
    "node_to_idx": global_node_to_idx,
    "node_family": node_family,
}, "final_dgi_index_embeddings.pt")

print(z.shape)
print("saved final_dgi_index_embeddings.pt")

torch.Size([5773, 128])
saved final_dgi_index_embeddings.pt
